<a href="https://colab.research.google.com/github/Aksinhaa/ColabFold/blob/main/TGW_data_preprocessing_QC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This workshop is a joint collaboration between NCBS - Bangalore, India `(Uma Ramakrishnan, Laura Bertola, Devkant Singha, Vinti Nanda)`; University of Edinburgh UK `(Katerina Guschanski, German Hernandez Alonso)`; IISc - Bangalore, India `(Anubhab Khan, Akancha Sinha )`; IISER Mohali, India `(Kritika M. Garg)`; and Birbal Sahni Institute of Palaeosciences (BSIP) - Lucknow, India `(Niraj Rai)`.
Scripts written by: `German Hernandez Alonso`
Notebook compiled by: `Akancha Sinha`

# **Temporal Genomics Workshop**


 Ancient DNA (aDNA) analysis, presents unique challenges due to degradation, short fragment lengths, and characteristic damage patterns. This workflow let you explore a complete pipeline for processing aDNA sequencing data, from raw reads to population-level study. It integrates multiple bioinformatics tools to ensure accurate alignment, variant calling, and downstream analysis while accounting for the limitations of low-coverage data.

## Workflow Overview
* Perform quality assessment of raw reads using FastQC and summarize with MultiQC

* Trim adapters and low-quality bases using AdapterRemoval

* Map reads to a reference genome using BWA and process alignments with Samtools

* Assess DNA damage patterns and alignment quality using DamageProfiler

* Call SNPs using ANGSD and perform population analysis with PLINK and PCA visualization in R


This workshop is divided into different sections, and this particular page focuses on data preprocessing and quality control steps.
This step is very crucial while handling genomic dataset as including poor quality bases and adapter contamination may lead to false positives.

In bioinformatics workflows, multiple tools are used, each with specific dependencies and version requirements. Managing these manually can lead to compatibility issues. **Conda** is an open-source package and environment management system designed to simplify installing software, managing dependencies, and creating isolated environments for projects. Hence, the first step would be to install miniconda and create a working environment where the required tools can be installed.

In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env fastqc adapterremoval multiqc

"mkdir" command is used for creating a directory. We will be creating separate directories throughout the pipeline to keep the wokflow organised.

"cd" is for going to the required directory

"pwd" is to print the path of working directory

In [ ]:
%%bash
mkdir -p /content/workshop_data/{fastq_files,trimmed_reads}
cd /content/workshop_data
pwd

This step downloads the raw sequencing data (FASTQ files) for three samples Hoggar1, Niger, and Senegal-P into the working directory. The wget -c command retrieves each file from the online repository. These FASTQ files contain the raw DNA sequences along with their quality scores, which serve as the starting point for all downstream analyses. Finally, ls -lh is used to list and verify that all files have been successfully downloaded and to inspect their sizes.

In [ ]:
%%bash
cd /content/workshop_data/fastq_files


# Download
wget -c https://zenodo.org/records/19913591/files/Hoggar1_1.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Hoggar1_2.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Niger_1.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Niger_2.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Senegal-P_1.100k.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Senegal-P_2.100K.fastq.gz

ls -lh

This step activates the Conda environment (`ngs_env`) to ensure that all required tools, including FastQC, are available.

A directory is then created to store the quality control results, helping keep the workflow organized.

The script navigates to the folder containing the raw FASTQ files and runs FastQC on all compressed sequencing files. This generates individual reports assessing read quality, GC content, and potential issues such as adapter contamination before further processing.

This is a very important pre processing step to understand the quality of the raw reads before downstraem analysis.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

mkdir -p /content/workshop_data/fastqc_results

cd /content/workshop_data/fastq_files

fastqc *.fastq.gz -o /content/workshop_data/fastqc_results

Once we have acquired the fastqc reports, we are going to run another tool called MultiQC, which will combine all individual FastQC reports into a single, comprehensive summary. This makes it easier to compare quality metrics across all samples in one place.

 Finally, `ls *.html` confirms that the MultiQC report has been successfully generated.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/fastqc_results
multiqc .
ls *.html

In [ ]:
from IPython.display import display, HTML

with open("/content/workshop_data/fastqc_results/multiqc_report.html", "r") as f:
    display(HTML(f.read()))

After ananlysing the Multiqc reports, now we are aware of the poor quality bases and adapter contamination in our raw data. The next step would be eliminating them before further analysis, for that purpose we are going to use AdapterRemoval tool.

 The script iterates over each sample name (Hoggar1, Niger, Senegal-P) and processes paired-end FASTQ files (`_1` and `_2`) together. The `--basename` option defines the output location and prefix, while `--gzip` compresses results and `--threads 2` speeds up execution using two CPU cores.


 `--trimns` and `--trimqualities` remove ambiguous bases and low-quality ends, while `--minquality 20` ensures only reliable bases are retained and `--minlength 30` filters out very short fragments.

  The adapter sequences (`--adapter1` and `--adapter2`) are explicitly provided so the tool can accurately remove sequencing adapters that are common in short aDNA reads.
  
  The `--minadapteroverlap 1` makes adapter detection sensitive, which is important for degraded fragments.

A key feature here is `--collapse`, which merges overlapping paired-end reads into a single sequence—this is especially important for aDNA because fragments are often shorter than the read length and overlap significantly.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/fastq_files

for sample in Hoggar1 Niger Senegal-P
do
  echo "Processing $sample..."

  AdapterRemoval \
  --file1 ${sample}_1.100*.fastq.gz \
  --file2 ${sample}_2.100*.fastq.gz \
  --basename /content/workshop_data/trimmed_reads/${sample} \
  --gzip \
  --threads 2 \
  --qualitymax 41 \
  --collapse \
  --trimns \
  --trimqualities \
  --adapter1 AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC \
  --adapter2 AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGTA \
  --minlength 30 \
  --minquality 20 \
  --minadapteroverlap 1

done

This step combines all processed read outputs from AdapterRemoval into a single file for each sample. Different categories of reads (collapsed, truncated, singleton, and paired reads) are concatenated using the cat command to ensure usable data is not lost. This creates one unified FASTQ file per sample, simplifying downstream alignment. Finally, ls -lh verifies that the combined files were successfully created and shows their sizes.

In [ ]:
%%bash
cd /content/workshop_data/trimmed_reads

for sample in Hoggar1 Niger Senegal-P
do
  echo "Combining files for $sample..."

  cat ${sample}.collapsed.gz \
      ${sample}.collapsed.truncated.gz \
      ${sample}.singleton.truncated.gz \
      ${sample}.pair1.truncated.gz \
      ${sample}.pair2.truncated.gz \
      > ${sample}.trimmed_combined.fastq.gz

done
ls -lh *.trimmed_combined.fastq.gz